In [197]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [198]:
df = pd.read_csv("diabetes_prediction_dataset.csv")
df.head(7)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
5,Female,20.0,0,0,never,27.32,6.6,85,0
6,Female,44.0,0,0,never,19.31,6.5,200,1


In [199]:
print("Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["diabetes"].value_counts())

Shape: (100000, 9)

Data Types:
gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object

Missing Values:
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Target Distribution:
diabetes
0    91500
1     8500
Name: count, dtype: int64


In [200]:
corr = df.corr(numeric_only=True)

corr["diabetes"].sort_values()

,diabetes
heart_disease,0.171727
hypertension,0.197823
bmi,0.214357
age,0.258008
HbA1c_level,0.400660
blood_glucose_level,0.419558
diabetes,1.000000


In [201]:
from scipy.stats import pearsonr

r, p = pearsonr(df["heart_disease"], df["diabetes"])

print("Correlation:", r)
print("P-value:", p)

Correlation: 0.17172684954884415
P-value: 0.0


In [202]:
from scipy.stats import f_oneway

groups = [
    group["diabetes"].values
    for _, group in df.groupby("smoking_history")
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 399.00023216135503
P-value: 0.0


In [203]:
df.head(7)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
5,Female,20.0,0,0,never,27.32,6.6,85,0
6,Female,44.0,0,0,never,19.31,6.5,200,1


In [204]:
df["gender"] = df["gender"].fillna("Unknown")
df["smoking_history"] = df["smoking_history"].fillna("Unknown")
df["gender"] = df["gender"].map({
    "Female": 0,
    "Male": 1,
    "Other": 2,
    "Unknown": 3
})

df["smoking_history"] = df["smoking_history"].map({
    "never": 0,
    "No Info": 1,
    "current": 2,
    "former": 3,
    "not current": 4,
    "Unknown": 5
})


In [205]:
X = df.drop(columns=["diabetes"])
y = df["diabetes"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100000, 8)
y shape: (100000,)


In [206]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=2,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 80000
Testing samples: 20000


In [207]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=2,
    stratify=y_train
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 64000
Validation samples: 16000
Testing samples: 20000


In [225]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Ensure X_train, X_val, X_test are numpy arrays for consistent operations
if isinstance(X_train, pd.DataFrame) or isinstance(X_train, pd.Series):
    X_train = X_train.to_numpy()
if isinstance(X_val, pd.DataFrame) or isinstance(X_val, pd.Series):
    X_val = X_val.to_numpy()
if isinstance(X_test, pd.DataFrame) or isinstance(X_test, pd.Series):
    X_test = X_test.to_numpy()

# Check for and replace infinite values with NaN
if np.isinf(X_train).any():
    print("Infinite values found in X_train. Replacing with NaN for imputation.")
    X_train[np.isinf(X_train)] = np.nan
if np.isinf(X_val).any():
    print("Infinite values found in X_val. Replacing with NaN for imputation.")
    X_val[np.isinf(X_val)] = np.nan
if np.isinf(X_test).any():
    print("Infinite values found in X_test. Replacing with NaN for imputation.")
    X_test[np.isinf(X_test)] = np.nan

# Impute any NaN values (either original or from replacing inf)
if np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any():
    print("NaN values detected. Imputing with SimpleImputer (mean strategy).")
    imputer = SimpleImputer(strategy='mean')

    # Fit imputer on training data and transform all splits
    X_train = imputer.fit_transform(X_train)
    X_val = imputer.transform(X_val)
    X_test = imputer.transform(X_test)
else:
    print("No NaN values to impute after checking for infinities.")

# Now apply StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Final check for NaNs after scaling
if np.isnan(X_train).any():
    print("FINAL CHECK: NaN detected in X_train after scaling!")
if np.isnan(X_val).any():
    print("FINAL CHECK: NaN detected in X_val after scaling!")
if np.isnan(X_test).any():
    print("FINAL CHECK: NaN detected in X_test after scaling!")

NaN values detected. Imputing with SimpleImputer (mean strategy).


In [236]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        if isinstance(features, (pd.DataFrame, pd.Series)):
            features = features.to_numpy()

        if isinstance(labels, (pd.DataFrame, pd.Series)):
            labels = labels.to_numpy()

        self.features = torch.tensor(
            features,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        ).reshape(-1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [227]:
train_dataset = CustomDataset(X_train, y_train)
val_dataset = CustomDataset(X_val, y_val)
test_dataset = CustomDataset(X_test, y_test)

In [228]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [212]:
X_batch, y_batch = next(iter(train_loader))

print("Feature batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)

Feature batch shape: torch.Size([32, 8])
Label batch shape: torch.Size([32, 1])


In [213]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [235]:
class DiabetesClassifier(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)

        return x

In [231]:
input_size = X_train.shape[1]

model = DiabetesClassifier(input_size)

model = model.to(device)

print(model)

DiabetesClassifier(
  (fc1): Linear(in_features=8, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (fc4): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)


In [216]:
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()

pos_weight_value = num_negative / num_positive

print("Negative samples:", num_negative)
print("Positive samples:", num_positive)
print("pos_weight:", pos_weight_value)

Negative samples: 58560
Positive samples: 5440
pos_weight: 10.764705882352942


In [217]:
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()

pos_weight_value = num_negative / num_positive

print("Negative samples:", num_negative)
print("Positive samples:", num_positive)
print("pos_weight:", pos_weight_value)

Negative samples: 58560
Positive samples: 5440
pos_weight: 10.764705882352942


In [232]:
pos_weight = torch.tensor(pos_weight_value, device=device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [237]:
epochs = 15

train_losses = []
val_losses = []

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================

    model.train()

    total_train_loss = 0.0

    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        outputs = model(batch_features)

        # Clamp outputs to prevent extreme values leading to NaN
        outputs = torch.clamp(outputs, min=-10.0, max=10.0)

        # Calculate loss
        loss = criterion(outputs, batch_labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)


    # =========================
    # VALIDATION
    # =========================

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for batch_features, batch_labels in val_loader:

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)

            # Clamp outputs to prevent extreme values leading to NaN
            outputs = torch.clamp(outputs, min=-10.0, max=10.0)

            loss = criterion(outputs, batch_labels)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)


    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)


    # =========================
    # PRINT PROGRESS
    # =========================

    if (epoch + 1) % 5 == 0 or epoch == 0:

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"| Train Loss: {avg_train_loss:.4f} "
            f"| Val Loss: {avg_val_loss:.4f}"
        )

Epoch [1/15] | Train Loss: 0.4883 | Val Loss: 0.4441
Epoch [5/15] | Train Loss: 0.4695 | Val Loss: 0.4338
Epoch [10/15] | Train Loss: 0.4448 | Val Loss: 0.3992
Epoch [15/15] | Train Loss: 0.4293 | Val Loss: 0.3970
